# Retention Analysis

This problem is also considered in the SQL section, here we do it in Python

- Define each customer’s cohort_month = the month of their first rental.
- For every later rental, compute cohort_index = months since cohort_month (0 for the first month, 1 for the next month, …).
- For each cohort × index, count distinct active customers, divide by cohort size → retention %.

In [28]:
import psycopg2
import pandas as pd
import dotenv
dotenv.load_dotenv()
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="dvdrental",
    user="admin",
    password="lohraspco",
)

df = pd.read_sql("SELECT rental_date,customer_id FROM DVD.RENTAL", con=conn)

C:\Users\mamma\AppData\Local\Temp\ipykernel_37932\417087990.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT rental_date,customer_id FROM DVD.RENTAL", con=conn)


In [29]:
df["rental_month"] = df["rental_date"].dt.to_period("M")
df1 = df.groupby("customer_id")["rental_month"].min().to_frame("first_month")
df = pd.merge(df, df1, on="customer_id")

In [ ]:
df["cohort_index"] = (df["rental_month"] - df["first_month"]).apply(lambda x:x.n)

In [46]:
df2 = df[df["cohort_index"].eq(0)].copy()
df2 = df2[["first_month","customer_id"]].drop_duplicates()
df2 = df2.groupby("first_month")["customer_id"].count().to_frame("cohort_size")

In [53]:
df_reten = df.groupby(["cohort_index","first_month"])["customer_id"].nunique().to_frame("retention").reset_index()

In [57]:
df_final = pd.merge(df2,df_reten, on="first_month")
df_final["ret_rate"] = df_final["retention"]/df_final["cohort_size"]
df_final

,first_month,cohort_size,cohort_index,retention,ret_rate
0,2005-05,520,0,520,1.000000
1,2005-05,520,1,512,0.984615
2,2005-05,520,2,520,1.000000
3,2005-05,520,3,520,1.000000
4,2005-05,520,9,139,0.267308
5,2005-06,78,0,78,1.000000
6,2005-06,78,1,78,1.000000
7,2005-06,78,2,78,1.000000
8,2005-06,78,8,19,0.243590
9,2005-07,1,0,1,1.000000
